# Demucs 语音分离 - Pitt残差数据集

本notebook使用Demucs（Hybrid Transformer）模型进行音频源分离，提取vocals（人声）部分。
注意：Demucs会将所有人声（包括讲话和唱歌）都分类为vocals，因此适用于语音降噪。

## 1. 导入库和环境检查

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
import torchaudio

# 过滤警告
warnings.filterwarnings('ignore')

print(f"PyTorch版本: {torch.__version__}")
print(f"Torchaudio版本: {torchaudio.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
print(f"MPS可用: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

## 2. 配置路径

In [ ]:
# 输入和输出目录
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-Demucs')

# 获取文件列表
control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

print(f"Control组文件数: {len(control_files)}")
print(f"Dementia组文件数: {len(dementia_files)}")

## 3. 加载 Demucs 模型

In [ ]:
from demucs.pretrained import get_model
from demucs.apply import apply_model

# 加载预训练的 Demucs 模型
# 可选模型:
# - 'htdemucs': Hybrid Transformer Demucs（推荐，平衡速度和质量）
# - 'htdemucs_ft': Hybrid Transformer Demucs 精调版（最高质量，速度较慢）
# - 'htdemucs_6s': 6源分离（包含吉他和钢琴）
# 注意：Demucs主要用于音乐源分离，但它会将所有人声（讲话+唱歌）都归类为vocals
#       因此提取vocals可以实现语音降噪，去除背景噪音、音乐等干扰

model_name = 'htdemucs_ft'
print(f"加载模型: {model_name}...")

# 设置设备
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("使用 CUDA GPU")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("使用 Apple MPS")
else:
    device = torch.device('cpu')
    print("使用 CPU")

# 加载模型
model = get_model(model_name)
model.to(device)
model.eval()

print(f"✓ 模型加载完成")
print(f"模型采样率: {model.samplerate} Hz")
print(f"输出源: {model.sources}")

## 4. 显存管理辅助函数

In [ ]:
def clear_memory():
    """
    清理显存和内存
    支持 CUDA 和 MPS 后端
    """
    # 清理 Python 垃圾回收
    gc.collect()
    
    # 清理 PyTorch 缓存
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
        torch.mps.synchronize()
    
    # 短暂等待，确保清理完成
    time.sleep(0.1)


def get_memory_info():
    """
    获取显存使用信息（仅用于调试）
    """
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3  # GB
        reserved = torch.cuda.memory_reserved() / 1024**3
        return f"CUDA - 已分配: {allocated:.2f} GB, 已保留: {reserved:.2f} GB"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        allocated = torch.mps.current_allocated_memory() / 1024**3
        return f"MPS - 已分配: {allocated:.2f} GB"
    return "CPU - 无显存统计"

## 5. 定义音频源分离函数

In [ ]:
def denoise_audio(audio_path, model, device):
    """
    使用 Demucs 进行语音分离和增强
    从音频中提取vocals（人声）部分，去除背景噪音、音乐等干扰
    注意：Demucs将所有人声（讲话+唱歌）都归类为vocals
    
    Args:
        audio_path: 输入音频文件路径
        model: Demucs 模型实例
        device: 计算设备 (cuda/mps/cpu)
    
    Returns:
        vocals_audio: 提取的人声音频 numpy array（单声道）
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # Demucs 需要 2 通道（立体声）输入
    # 处理不同声道数的音频
    if len(audio.shape) == 1:
        # 单声道：复制为双声道 [time] -> [time, 2]
        audio = np.stack([audio, audio], axis=1)
    elif len(audio.shape) == 2:
        if audio.shape[1] == 1:
            # [time, 1] -> [time, 2]
            audio = np.concatenate([audio, audio], axis=1)
        elif audio.shape[1] > 2:
            # 多于2个通道，只取前2个
            audio = audio[:, :2]
        # 如果已经是2通道，保持不变
    
    # 重采样到模型要求的采样率
    target_sr = model.samplerate
    if sr != target_sr:
        # 对每个通道分别重采样
        num_samples = int(audio.shape[0] * target_sr / sr)
        audio_resampled = np.zeros((num_samples, 2), dtype=audio.dtype)
        for ch in range(2):
            audio_resampled[:, ch] = signal.resample(audio[:, ch], num_samples)
        audio = audio_resampled
        sr = target_sr
    
    # 转换为 torch tensor
    # Demucs 期望输入形状为 [batch, channels, time]
    # audio 当前形状: [time, 2]
    audio_tensor = torch.from_numpy(audio.T).float()  # [2, time]
    audio_tensor = audio_tensor.unsqueeze(0)  # [1, 2, time]
    audio_tensor = audio_tensor.to(device)
    
    # 应用 Demucs 源分离
    with torch.no_grad():
        # apply_model 返回分离后的音频源
        # 输出形状: [batch, sources, channels, time]
        # sources 顺序通常为: ['drums', 'bass', 'other', 'vocals']
        sources = apply_model(
            model, 
            audio_tensor, 
            device=device,
            split=True,  # 分段处理，节省显存
            overlap=0.25  # 重叠25%以避免边界效应
        )
    
    # 提取 vocals 源（包含所有人声：讲话+唱歌）
    # 找到 vocals 在源列表中的索引
    try:
        vocals_idx = model.sources.index('vocals')
    except (AttributeError, ValueError):
        # 如果找不到，假设是最后一个源
        vocals_idx = -1
    
    # 转换回 numpy，并转为单声道
    # sources 形状: [batch, sources, channels, time]
    # 取双通道的平均值作为单声道输出
    vocals_stereo = sources[0, vocals_idx, :, :].cpu().numpy()  # [2, time]
    vocals_audio = np.mean(vocals_stereo, axis=0)  # [time] 单声道
    
    return vocals_audio, sr

## 6. 定义批量源分离函数（带显存清理）

In [ ]:
def batch_denoise(files, output_subdir, model, device, group_name):
    """
    批量音频源分离处理（每个文件前后都清理显存）
    提取vocals（人声）部分，去除背景噪音
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: Demucs 模型实例
        device: 计算设备
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"源分离处理 {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            # ⚡ 处理前清理显存
            clear_memory()
            
            # 源分离，提取vocals（人声）
            vocals_audio, sr = denoise_audio(audio_file, model, device)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), vocals_audio, sr, subtype='PCM_16')
            success_count += 1
            
            # ⚡ 处理后立即清理显存
            del vocals_audio  # 删除大数组
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\n✗ 处理失败: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"  ✓ 成功: {success_count}")
    print(f"  ⊘ 跳过: {skip_count}")
    print(f"  ✗ 失败: {fail_count}")
    print(f"  Σ 总计: {len(files)}")

## 7. 执行批量源分离处理

In [ ]:
# 记录开始时间
start_time = time.time()

# ⚡ 开始前先清理显存
print("初始显存状态:", get_memory_info())
clear_memory()
print("清理后显存:", get_memory_info())

# 处理 Dementia 组
print("\n" + "="*60)
print("开始处理 Dementia 组")
print("="*60)
batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    model,
    device,
    group_name='Dementia'
)

# ⚡ 两组之间清理显存
clear_memory()
print("\n组间清理后显存:", get_memory_info())

# 处理 Control 组
print("\n" + "="*60)
print("开始处理 Control 组")
print("="*60)
batch_denoise(
    control_files,
    output_dir / 'Control',
    model,
    device,
    group_name='Control'
)

# 计算总时间
elapsed_time = time.time() - start_time
print("\n" + "="*60)
print(f"✓ 所有处理完成！")
print(f"总耗时: {elapsed_time/60:.2f} 分钟 ({elapsed_time:.2f} 秒)")
print(f"输出目录: {output_dir}")
print(f"最终显存状态: {get_memory_info()}")
print("="*60)